# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a practical template for loading and exploring the FAIR² dataset package using the `mlcroissant` library. The dataset contains ordered logistic regression outputs and associated survey data related to household adoption of indigenous and modern knowledge in rangeland management across Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset metadata (name and description)
print(f'Dataset Name: {metadata.name}')
print(f'Description: {metadata.description}')

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps you identify what data entities and variables are present for further exploration.

In [ ]:
# List the available record sets, referencing them by their @id
record_sets = [rs for rs in metadata.record_sets]

print('Available record sets and their @id:')
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`.

In [ ]:
# Prepare to extract all record sets
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record is a dictionary mapping field @id to value
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# As an example, let's print columns and a preview for the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Loaded DataFrame for record set @id: {main_rs_id}")
    print(f"Columns (field @id): {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> In this section, we'll select a numeric field from the main record set for analysis. You may choose another field or record set by updating the field `@id`.

In [ ]:
# Select a numeric field for exploration by its @id
# Replace with actual @id from step 2 if necessary
if record_set_ids:
    df = dataframes[main_rs_id]
    numeric_field_candidates = []
    # Try to select numeric fields automatically
    for rs in metadata.record_sets:
        if rs.id == main_rs_id:
            for f in rs.fields:
                if hasattr(f, "data_type") and f.data_type and ("Float" in f.data_type or "Integer" in f.data_type or "Number" in f.data_type):
                    numeric_field_candidates.append(f.id)
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]  # Pick the first numeric field by @id
        print(f"Using numeric field @id: {numeric_field_id}")
        # Ensure correct column exists and not all values are null
        if numeric_field_id in df.columns and df[numeric_field_id].notnull().any():
            # Convert column to numeric, errors='coerce' for safety
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean()  # Use mean as a dynamic threshold
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean)")
            print(filtered_df.head())
            # Normalize
            normalized_col = f"{numeric_field_id}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, normalized_col]].head())
            # Attempt groupby with a categorical field (e.g., first string field found)
            group_field_candidates = []
            for rs in metadata.record_sets:
                if rs.id == main_rs_id:
                    for f in rs.fields:
                        if hasattr(f, "data_type") and f.data_type and ("Text" in f.data_type):
                            group_field_candidates.append(f.id)
            if group_field_candidates:
                group_field = group_field_candidates[0]
                print(f"\nGrouping by field @id: {group_field}")
                if group_field in filtered_df.columns:
                    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                    print(grouped_df.head())
    else:
        print("No numeric field found in main record set. Please check available fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using common Python plotting libraries.

> Here we will plot a histogram for the selected numeric field and, if grouping is available, a bar plot of the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the selected numeric field
if record_set_ids and numeric_field_candidates:
    num_fld = numeric_field_candidates[0]
    # Ensure all NaNs are dropped for plotting
    valid_vals = df[num_fld].dropna()
    plt.figure(figsize=(8, 4))
    sns.histplot(valid_vals, kde=True, bins=20)
    plt.title(f'Distribution of {num_fld}')
    plt.xlabel(num_fld)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If grouping done in EDA above
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, you:
- Loaded dataset metadata and records from a Croissant schema.
- Explored the structure of available record sets and fields, referencing all entities by their unique `@id`.
- Loaded tabular data into pandas DataFrames using the `mlcroissant` library.
- Performed filtering, normalization, and grouping on a selected numeric field for exploratory analysis.
- Generated visualizations to better understand the distribution and group-level summaries of your chosen variables.

This approach is fully reproducible for any Croissant-compatible data package. Refer to the field and record set `@id`s when applying these steps to additional datasets!